# 03 · Build the architecture and inspect the experiment matrix


In [ ]:
from pathlib import Path
import json, os, sys

# Find the checkout/release from the notebook's working directory.
ROOT = Path(os.environ.get('GF_ROOT', Path.cwd())).resolve()
while not (ROOT / 'src/gavd6_sjepa').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/gavd6_sjepa').is_dir(), 'Open this notebook from the GAVD6 checkout or release.'
sys.path.insert(0, str(ROOT / 'notebooks/gait_fidelity'))
sys.path.insert(0, str(ROOT / 'src'))
from tutorial_helpers import configure, preview_images
study = configure(ROOT)


## From observed coordinates to a restored trajectory

The production model consumes a fixed window of 12 body joints. For each joint
and frame it receives `(x, y)`, the estimator's confidence, an observed flag,
and elapsed time in seconds. Reference coordinates enter training losses and
the training-only teacher; they never enter the restoration model's input.

The encoder groups four consecutive frames of **one joint** into a token.
For a window of $T$ frames, patch length $P$, 12 joints, and feature width $D$,
the shapes are

$$[B,T,12,5]\longrightarrow[B,T/P,12,5P]
  \longrightarrow[B,12T/P,D].$$

We will trace a fresh, small CPU model with $T=16$, $P=4$, $D=16$, and two
examples. The saved source study normally uses 128 frames and width 96;
the table below reads your actual configuration. This local numerical example
uses the same operators and parameter layout, with fewer tokens and layers.
Its random weights and generated trajectories provide software checks only.

This is the repository's body-12 restoration architecture. The
multiple-sclerosis tutorials provide the explanatory style, but their
33-landmark model and classification task are different implementations.


In [ ]:
import math
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display
from gavd6_sjepa.research_directions.synthetic_training_v2.models import ModelConfig, RestorationModel
from gavd6_sjepa.research_directions.gait_fidelity.training import normalize_batch

saved_config = study.artifact('config.json')
small = ModelConfig(width=16, encoder_layers=1, predictor_layers=1,
                    heads=2, patch_size=4, window_size=16)
display(pd.DataFrame({'saved_study': saved_config['model'], 'teaching_example': vars(small)}))

# Each construction restores the caller's random state on exit.
torch_state_before = torch.random.get_rng_state().clone()
with torch.random.fork_rng(devices=[]):
    torch.random.default_generator.manual_seed(601)
    scratch = RestorationModel('paired_jepa', small).cpu().eval()
assert torch.equal(torch_state_before, torch.random.get_rng_state())

B, T, J, P, D = 2, small.window_size, 12, small.patch_size, small.width
S = T // P
template = np.array([[-25,20],[25,20],[-35,55],[35,55],[-40,85],[40,85],
                     [-20,95],[20,95],[-20,145],[20,145],[-20,195],[20,195]], np.float32)
seconds = np.arange(T, dtype=np.float32) / 25
reference_px = np.broadcast_to(template, (B,T,J,2)).copy()
reference_px[:, :, 8, 0] += 12 * np.sin(2*np.pi*seconds)[None]
reference_px[:, :, 9, 0] += 16 * np.sin(2*np.pi*seconds + .3)[None]
observed_px = reference_px + np.array([3., -2.], np.float32)
observed = np.ones((B,T,J), bool)
observed[0, :P, 10] = False
observed_px[~observed] = np.nan
raw = dict(xy=observed_px, confidence=np.full((B,T,J), .8, np.float32),
           observed=observed, timestamps=np.broadcast_to(seconds, (B,T)).copy())
hidden_np = np.zeros((B,S,J), bool)
hidden_np[:, 2, 8:10] = True
normalized, origin, scale, fallbacks = normalize_batch(raw, hidden_np, patch_size=P)
inputs = {key: torch.as_tensor(value) for key, value in normalized.items()}
hidden = torch.as_tensor(hidden_np)
print({'example_scope': 'generated CPU teaching tensors',
       'input_shape': tuple(inputs['xy'].shape), 'context_normalization_fallbacks': fallbacks})


## Pack the five channels without leaking hidden coordinates

Artificial masks are repeated across the four frames of their patch. A usable
coordinate must be observed and not artificially hidden. Hidden coordinates
and their confidence become zero; elapsed time remains available.

Natural absence has a separate convention: its coordinates and observed flag
are zeroed, while a finite native confidence is retained unless that slot is
also artificially hidden. Thus confidence and visibility are distinct inputs.
The input-only normalization computed above excludes artificially hidden
coordinates before calculating its origin and scale, as notebook 01 explains.

The following code reproduces the production channel order and token order.
Token index `patch * 12 + joint` identifies one joint in one temporal patch.


In [ ]:
artificial = hidden.repeat_interleave(P, dim=1)
usable = inputs['observed'] & ~artificial
safe_xy = torch.where(usable[..., None], inputs['xy'], 0)
confidence = torch.where(~artificial, inputs['confidence'], 0)
elapsed = inputs['timestamps'] - inputs['timestamps'][:, :1]
channels = torch.cat([safe_xy, confidence[..., None], usable[..., None].float(),
                      elapsed[:, :, None, None].expand(-1, -1, J, -1)], dim=-1)
patches = channels.reshape(B,S,P,J,5).permute(0,1,3,2,4).reshape(B,S*J,P*5)
assert torch.equal(patches[0, 2*J+8], channels[0, 2*P:3*P, 8].flatten())
assert torch.isfinite(patches).all()
print('Channels:', tuple(channels.shape), 'Patch vectors:', tuple(patches.shape))
display(pd.DataFrame(channels[0, 2*P:3*P, 8].numpy(),
                     columns=['x', 'y', 'confidence', 'usable', 'elapsed_seconds']))


## Add position and joint identity

A linear projection maps the $5P$ numbers into $D$ learned features. The model
adds a fixed sinusoidal encoding of the patch index and a learned embedding
of the joint identity. Physical seconds remain in the input channels; patch
indices supply a separate ordered query grid.

$$z_{s,j}=W\,\mathrm{patch}_{s,j}+b+e^{\mathrm{time}}_s
          +e^{\mathrm{joint}}_j
          +\mathbf{1}[\text{no usable frame}]\,e^{\mathrm{missing}}.$$

All patch–joint slots remain in the sequence, including missing slots. The
learned missing-query vector starts at zero. A missing slot can attend to
other slots and eventually receive a prediction; it is not removed by an
attention padding mask.


In [ ]:
patch_number = torch.arange(S, dtype=torch.float32)[:, None]
frequency = torch.exp(torch.arange(0, D, 2).float() * (-math.log(10000.) / D))
position = torch.zeros(S, D)
position[:, 0::2] = torch.sin(patch_number * frequency)
position[:, 1::2] = torch.cos(patch_number * frequency[:D//2])
patch_supported = usable.reshape(B,S,P,J).any(dim=2).reshape(B,S*J)
tokens = F.linear(patches, scratch.encoder.projection.weight, scratch.encoder.projection.bias)
tokens = tokens + (position[:, None] + scratch.encoder.joints.weight[None]).reshape(1,S*J,D)
tokens = tokens + (~patch_supported)[..., None] * scratch.encoder.missing_query
print({'tokens': tuple(tokens.shape),
       'entirely_missing_patch_slots': int((~patch_supported).sum()),
       'all_slots_retained': tokens.shape[1] == S*J})


## Calculate one self-attention layer explicitly

Each attention head makes query, key, and value vectors from normalized
tokens. Their dot products determine the weights used to combine values:

$$Q=XW_Q^\top+b_Q,\quad K=XW_K^\top+b_K,\quad V=XW_V^\top+b_V,$$
$$\mathrm{Attention}(X)=\mathrm{softmax}(QK^\top/\sqrt{d_h})V.$$

The production Transformer uses normalization **before** each sublayer, a
residual addition around attention, and another around its GELU feed-forward
network. The feed-forward hidden width is $4D$ and dropout is zero. Attention
uses the full window in both temporal directions, so this is offline
restoration rather than causal forecasting.


In [ ]:
layer = scratch.encoder.blocks.layers[0]
normalized_tokens = layer.norm1(tokens)
q, k, v = F.linear(normalized_tokens, layer.self_attn.in_proj_weight,
                    layer.self_attn.in_proj_bias).chunk(3, dim=-1)
H, Dh, N = small.heads, D // small.heads, S * J
split_heads = lambda a: a.reshape(B,N,H,Dh).transpose(1,2)
q, k, v = map(split_heads, (q,k,v))
attention_weights = (q @ k.transpose(-2,-1) / math.sqrt(Dh)).softmax(dim=-1)
context = (attention_weights @ v).transpose(1,2).reshape(B,N,D)
attention_output = F.linear(context, layer.self_attn.out_proj.weight,
                            layer.self_attn.out_proj.bias)
after_attention = tokens + attention_output
feedforward = layer.linear2(F.gelu(layer.linear1(layer.norm2(after_attention))))
after_layer = after_attention + feedforward
torch.testing.assert_close(after_layer, layer(tokens), atol=2e-6, rtol=2e-5)
torch.testing.assert_close(attention_weights.sum(-1), torch.ones(B,H,N))

encoded = scratch.encoder.norm(after_layer)
torch.testing.assert_close(encoded, scratch.encoder(inputs, hidden), atol=2e-6, rtol=2e-5)
print('Explicit attention and encoder match production:', tuple(encoded.shape))


In [ ]:
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import Image
fig, ax = plt.subplots(figsize=(6,5), constrained_layout=True)
plot = ax.imshow(attention_weights[0].mean(0).detach().numpy(), aspect='equal', cmap='viridis')
ax.set(xlabel='Key token (patch × 12 + joint)', ylabel='Query token (patch × 12 + joint)',
       title='Generated CPU example: random attention')
fig.colorbar(plot, ax=ax, label='Attention weight (head mean)')
buffer = BytesIO(); fig.savefig(buffer, format='png', dpi=120)
display(Image(data=buffer.getvalue())); plt.close(fig)
print('These weights illustrate the operator; they are not learned anatomical importance.')


## Decode features into coordinates

The output network applies LayerNorm, a linear layer, GELU, and a final linear
layer that emits $2P$ coordinate corrections per token. Reshaping restores the
frame order. At observed locations the network predicts a residual added to
the input coordinate; at missing locations it predicts an absolute normalized
coordinate, with zero as the addition base.

$$\widehat{x}_{t,j}=r_{t,j}+
  \begin{cases}x_{t,j},&\text{usable input};\\0,&\text{otherwise.}\end{cases}$$

The final linear layer starts at zero. Thus this fresh model initially passes
through usable observations and emits the normalization origin in pixel space
at missing slots. Those missing predictions can be inaccurate; the loss and
evaluation retain their errors.


In [ ]:
readout_layers = scratch.readout.network
patch_correction = readout_layers[3](F.gelu(readout_layers[1](readout_layers[0](encoded))))
correction = patch_correction.reshape(B,S,J,P,2).permute(0,1,3,2,4).reshape(B,T,J,2)
restored_normalized = correction + torch.where(usable[..., None], inputs['xy'], 0)
torch.testing.assert_close(restored_normalized, scratch(inputs, hidden), atol=2e-6, rtol=2e-5)
assert torch.count_nonzero(correction) == 0
restored_pixels = restored_normalized.detach().numpy() * scale[:,None,None,None] + origin[:,None,None]
np.testing.assert_allclose(restored_pixels[usable.numpy()], observed_px[usable.numpy()], atol=2e-5)
print({'feature_shape': tuple(encoded.shape), 'prediction_shape': tuple(restored_pixels.shape),
       'initial_correction_maximum': float(correction.abs().max().detach())})


The numerical derivation used a small model to keep the tensors readable. We
also check the source study's default dimensions on CPU: 128 frames form
384 patch–joint tokens, each with 96 features. This shape check uses another
fresh model and has no bearing on trained accuracy or HAIC GPU compatibility.


In [ ]:
source_dimensions = ModelConfig(window_size=128)
with torch.random.fork_rng(devices=[]):
    torch.random.default_generator.manual_seed(602)
    shape_model = RestorationModel('direct', source_dimensions).cpu().eval()
shape_inputs = dict(xy=torch.as_tensor(template)[None,None].expand(1,128,12,2)/200,
                    confidence=torch.ones(1,128,12), observed=torch.ones(1,128,12,dtype=torch.bool),
                    timestamps=torch.arange(128,dtype=torch.float32)[None]/25)
with torch.no_grad():
    source_features = shape_model.encoder(shape_inputs)
    source_output = shape_model(shape_inputs)
assert source_features.shape == (1,384,96) and source_output.shape == (1,128,12,2)
assert torch.isfinite(source_output).all()
assert torch.equal(torch_state_before, torch.random.get_rng_state())
print('Default source shapes:', tuple(source_features.shape), '→', tuple(source_output.shape))
del shape_model


## Connect the architecture to the experiment matrix

Coordinate pretraining fits the encoder and coordinate readout at queried
patches. Paired JEPA fits an encoder and feature predictor against an
exponential-moving-average teacher that sees matching reference trajectories.
Its predictor produces $D$ features per token and is distinct from the
coordinate readout. Notebook 04 derives both losses and executes the updates.

| Phase or model | Updated parameters | Inputs in this phase |
| --- | --- | --- |
| Coordinate pretraining | Encoder and coordinate readout | Masked observations |
| Paired or shuffled-reference JEPA pretraining | Encoder, feature predictor, and regularization projector; teacher by moving average | Masked observations |
| Frozen-feature output training | A freshly initialized coordinate readout | Ordinary observed inputs, without an artificial pretraining mask |
| Direct training | Encoder and coordinate readout together | Ordinary observed inputs |
| Initialized encoder control | A new readout over fixed random encoder features | Ordinary observed inputs |
| Static benchmark | Framewise coordinate network plus full-window confidence/visibility/time context | Current-frame coordinates and shared auxiliary context |
| Temporal refiner | Shared per-joint temporal MLP | Each joint's full-window channels |

The static benchmark removes neighboring-frame **coordinates**, while retaining
full-window auxiliary information. The temporal refiner is a local 2D
SmoothNet-style adaptation; the study does not claim to reproduce author
weights or the original benchmark. These practical models are explained further
in experiment C.

The saved matrix fixes every comparison before ranking results. Three seeds
measure training variability; they do not add independent evaluation people.


In [ ]:
study.command('plan', quiet=True)
plan = study.artifact('plan.json')
recipes = pd.DataFrame(plan['recipes'])
phases = pd.DataFrame(plan['phases'])
print(json.dumps(plan['counts'], indent=2))
display(recipes)
display(phases[['phase_id', 'phase', 'seed', 'depends_on']].head(20))
assert len(recipes) == plan['counts']['recipes']
assert plan['counts']['final_fits'] == len(recipes) * len(plan['seeds'])


Identical pretraining can supply several independently fitted output networks
only when its data, mask, objective and seed agree. The three-seed core plan
contains nine pretraining phases, 24 frozen-feature output phases and six
end-to-end phases, producing 30 final models. The full plan contains 33
pretraining phases and 102 final models. The table above reports the selected
plan, so results from a core run cannot imply that omitted controls were fitted.

The source schedule starts at 2,000 updates per pretraining or output phase and
4,000 for end-to-end training. Hardware profiling can select the declared
common fallback before final fits. The scratch calculations above do not
change that plan, its random states, or its checkpoints. Continue to
**04_run_and_monitor.ipynb** to calculate the losses and follow optimization.
